**SparkSession**

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, avg, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Tugas7_Dany_Akhdan_Imaduddin") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat, Versi", spark.version)

SparkSession berhasil dibuat, Versi 4.2.0


**A & B. EXTRACT & TRANSFORM (Join)**

In [4]:
# A. EXTRACT
df_trx_kopi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv", header=True, inferSchema=True)
df_cabang_kopi = spark.read.json("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json")
df_menu_kopi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv", header=True, inferSchema=True)

# B. TRANSFORM (Join)
df_kopi = df_trx_kopi.join(df_cabang_kopi, on="cabang_id", how="inner") \
                     .join(df_menu_kopi, on="menu_id", how="inner")

df_kopi = df_kopi.withColumn("total_penjualan", col("qty") * col("harga"))
df_kopi.show(5)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


**C. TRANSFORM (Window Function Top-2 Menu)**

In [5]:
ringkasan_menu = df_kopi.groupBy("nama_cabang", "nama_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan_menu")
)

window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan_menu").desc())
top2_menu = ringkasan_menu.withColumn("peringkat", row_number().over(window_cabang)) \
                          .filter(col("peringkat") <= 2)

top2_menu.orderBy("nama_cabang", "peringkat").show()

+--------------------+---------+--------------------+---------+
|         nama_cabang|nama_menu|total_penjualan_menu|peringkat|
+--------------------+---------+--------------------+---------+
|Kopi Nusantara Ma...|   Menu-1|             8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|             8085000|        2|
|Kopi Nusantara Se...|  Menu-13|             8750000|        1|
|Kopi Nusantara Se...|   Menu-9|             8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|             9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|             8160000|        2|
+--------------------+---------+--------------------+---------+



**D. TRANSFORM (Spark SQL Barista Contribution)**

In [6]:
df_kopi.createOrReplaceTempView("data_kopi")
spark.sql('''
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan_barista
    FROM data_kopi
    GROUP BY kepala_barista
    ORDER BY total_penjualan_barista DESC
''').show()

+--------------+-----------------------+
|kepala_barista|total_penjualan_barista|
+--------------+-----------------------+
|         Nadia|              177680000|
|          Reza|               84645000|
+--------------+-----------------------+



**E. LOAD to Partitioned Parquet HDFS**

In [7]:
output_path = "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"
df_kopi.write.mode("overwrite").partitionBy("kategori_menu").parquet(output_path)

df_load = spark.read.parquet(output_path)
print("Total baris tersimpan di HDFS:", df_load.count())

Total baris tersimpan di HDFS: 4000


**F. Executive Summary & Recommendations**

In [8]:
ringkasan_kopi = df_kopi.groupBy("nama_cabang").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan").desc())

ringkasan_kopi.show()

+--------------------+---------------+----------------+-------------------------+
|         nama_cabang|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+--------------------+---------------+----------------+-------------------------+
|Kopi Nusantara Se...|       90542000|            1393|         64997.8463747308|
|Kopi Nusantara Ma...|       87138000|            1302|        66926.26728110599|
|Kopi Nusantara Yo...|       84645000|            1305|        64862.06896551724|
+--------------------+---------------+----------------+-------------------------+



Berdasarkan kompilasi data dari laporan *executive summary*, **Kopi Nusantara Magelang** berhasil membuktikan diri sebagai cabang dengan kinerja operasional paling unggul. Cabang ini sukses mencatatkan akumulasi nilai transaksi tertinggi sekaligus mempertahankan volume kedatangan pelanggan yang paling padat dibandingkan cabang lain. Di sisi lain, **Kopi Nusantara Semarang** justru menunjukkan tren yang berkebalikan. Cabang ini membukukan tingkat konversi harian dan total pendapatan paling rendah di antara seluruh jaringan kedai, sehingga menuntut adanya intervensi strategis dan perbaikan manajerial secepat mungkin agar tidak membebani target triwulanan perusahaan secara keseluruhan.

Untuk memulihkan dan mendongkrak angka penjualan di area Semarang, tim manajemen sangat disarankan untuk mengalokasikan anggaran pemasaran secara lebih agresif dan terarah guna mempromosikan **Menu-13**. Analisis data membuktikan secara empiris bahwa menu tersebut merupakan penggerak pendapatan utama (*revenue driver*) dengan daya tarik terkuat bagi basis konsumen lokal di Semarang. Sebagai langkah taktis lanjutan, pihak kedai dapat merancang program diskon atau *bundling* yang memasangkan Menu-13 dengan produk pelengkap lainnya yang kurang laris. Strategi *cross-selling* ini diharapkan tidak hanya mampu menarik kembali minat beli pelanggan reguler, tetapi juga secara efektif memperlebar jangkauan pasar dan menstabilkan arus kas cabang Semarang dalam jangka panjang.

**End SparkSession**

In [9]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
